# Mental Health in the Tech Industry
## CS 210: Data Management for Data Science
**Authors:** Vanij Patel and Sean Lumasag

**Research Question:** What workplace and personal factors predict whether a tech worker will seek mental health treatment, and how does mental health interfere with their work?

**Datasets:** OSMI Mental Health in Tech Surveys — 2014 (1,259 rows) and 2016 (1,433 rows)

In [ ]:
#imported libraries
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, f1_score, accuracy_score)
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

os.makedirs('output_figures', exist_ok=True)

BLUE   = '#4C72B0'
ORANGE = '#DD8452'
GREEN  = '#2ca02c'

print("All libraries loaded.")

/usr/local/bin/python3
All libraries loaded.


In [ ]:
#read the csv files and stored them
df14 = pd.read_csv('2014.csv')
df16 = pd.read_csv('2016.csv')

#load the data into sqlite database
connect = sqlite3.connect('mental_health.db')
df14.to_sql('mental_health_2014', connect, if_exists='replace', index=False)
df16.to_sql('mental_health_2016', connect, if_exists='replace', index=False)
print("Data loaded into SQLite database.")

#print the number of rows and columns in each dataframe
print(f"2014: {df14.shape[0]} rows, {df14.shape[1]} columns")
print(f"2016: {df16.shape[0]} rows, {df16.shape[1]} columns")
print("\nSaved to 'mental_health.db'.")

Data loaded into SQLite database.
2014: 1259 rows, 27 columns
2016: 1433 rows, 63 columns

Saved to 'mental_health.db'.


In [3]:
#check for null values in each dataframe and print columns with >20% nulls
null14 = (df14.isnull().sum() / len(df14) * 100).round(1)
print("2014 Columns with >20% Nulls:")
print(null14[null14 > 20].sort_values(ascending=False))

2014 Columns with >20% Nulls:
comments          87.0
state             40.9
work_interfere    21.0
dtype: float64


In [4]:
#check for null values in each dataframe and print columns with >20% nulls
null16 = (df16.isnull().sum() / len(df16) * 100).round(1)
print("2016 Columns with >20% Nulls:")
print(null16[null16 > 20].sort_values(ascending=False))

2016 Columns with >20% Nulls:
If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?                                                        90.0
If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?                                                85.8
Is your primary role within your company related to tech/IT?                                                                                                                        81.6
Do you know local or online resources to seek help for a mental health disorder?                                                                                                    80.0
If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?                                                        80.0
Do you have medical coverage (private insuran

In [5]:
#print the column names of each dataframe
print("2014 columns:", list(df14.columns))
print(f"\n2016 has {len(df16.columns)} columns written as longer questions.")
print("Example 2016 column:", df16.columns[0])

2014 columns: ['Timestamp', 'Age', 'Gender', 'Country', 'state', 'self_employed', 'family_history', 'treatment', 'work_interfere', 'no_employees', 'remote_work', 'tech_company', 'benefits', 'care_options', 'wellness_program', 'seek_help', 'anonymity', 'leave', 'mental_health_consequence', 'phys_health_consequence', 'coworkers', 'supervisor', 'mental_health_interview', 'phys_health_interview', 'mental_vs_physical', 'obs_consequence', 'comments']

2016 has 63 columns written as longer questions.
Example 2016 column: Are you self-employed?


In [6]:
# Manual mapping: 2016 question text → 2014 column name
SCHEMA_MAP = {
    'Are you self-employed?': 'self_employed',
    'How many employees does your company or organization have?': 'no_employees',
    'Is your employer primarily a tech company/organization?': 'tech_company',
    'Does your employer provide mental health benefits as part of healthcare coverage?': 'benefits',
    'Do you know the options for mental health care available under your employer-provided coverage?': 'care_options',
    'Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?': 'wellness_program',
    'Does your employer offer resources to learn more about mental health concerns and options for seeking help?': 'seek_help',
    'Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?': 'anonymity',
    'If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:': 'leave',
    'Do you think that discussing a mental health disorder with your employer would have negative consequences?': 'mental_health_consequence',
    'Do you think that discussing a physical health issue with your employer would have negative consequences?': 'phys_health_consequence',
    'Would you feel comfortable discussing a mental health disorder with your coworkers?': 'coworkers',
    'Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?': 'supervisor',
    'Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?': 'obs_consequence',
    'Would you bring up a mental health issue with a potential employer in an interview?': 'mental_health_interview',
    'Would you be willing to bring up a physical health issue with a potential employer in an interview?': 'phys_health_interview',
    'Do you feel that your employer takes mental health as seriously as physical health?': 'mental_vs_physical',
    'Do you have a family history of mental illness?': 'family_history',
    'Have you ever sought treatment for a mental health issue from a mental health professional?': 'treatment',
    'If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?': 'wi_treated',
    'If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?': 'wi_untreated',
    'What is your age?': 'Age',
    'What is your gender?': 'Gender',
    'What country do you live in?': 'Country',
    'What US state or territory do you live in?': 'state',
    'Do you work remotely?': 'remote_work',
    'How willing would you be to share with friends and family that you have a mental illness?': 'share_friends_family',
}

# Align 2016 dataset to 2014 schema using the mapping
df16_aligned = df16.rename(columns=SCHEMA_MAP).copy()
df16_aligned['survey_year'] = 2016

df14_aligned = df14.copy()
df14_aligned['survey_year'] = 2014

print("Mapped " + str(len(SCHEMA_MAP)) + " columns from 2016 question text to 2014 names.")
print("survey_year column added to both datasets.")

Mapped 27 columns from 2016 question text to 2014 names.
survey_year column added to both datasets.


In [ ]:
#wrote a function to clean the Age column
def clean_age(df):
    df = df.copy()
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df.loc[(df['Age'] < 18) | (df['Age'] > 75), 'Age'] = np.nan
    df['Age'] = df['Age'].fillna(df['Age'].median())
    return df

#clean the Age column in both datasets
df14_clean_age = clean_age(df14_aligned)
df16_clean_age = clean_age(df16_aligned)

print("Age column cleaned. Ages are between 18 and 75. Anything else is filled with median.")